# Prototype recurrence beyond classical ECG interpretation

`proto-recur-multilabel.ipynb` shows that PTB-XL, CinC and CODE-15% reuse the same
pretrained prototypes for the same rhythm and conduction labels. Those labels are
read off the ECG itself, so recurrence there is reassuring but expected.

This notebook asks the same question for labels that are **not** ECG reads: echo
findings in EchoNext and discharge diagnoses (phecodes) in MIMIC-IV-ECG. Two label
pairs describe the same disease in both datasets:

| Concept | EchoNext (echo) | MIMIC (ICD-10 via phecode 1.2) |
|---|---|---|
| LV systolic dysfunction | `LVEF Lo` (LVEF ≤ 45%) | HFrEF (phecode 428.3) |
| Aortic valve disease | `AS` (moderate or worse stenosis) | nonrheumatic aortic valve disorders (phecode 395.2) |

The pretrained prototypes carry no labels, so if both datasets pick the same
prototypes for these labels, the bank encodes the disease and not one dataset's way of
labeling it. The classical datasets (PTB-XL, CinC, CODE-15%) also assign every
prototype to a label of their own, which tells us what ECG finding each recurrent
prototype stands for.

The notebook produces:

1. a screen of every EchoNext × MIMIC label pair (over all seeds), showing these two
   pairs are the outliers rather than cherry-picked,
2. the stability of the shared prototypes across seeds, and Venn diagrams at `SEED`,
3. what each shared prototype means in PTB-XL, CinC and CODE-15%, and which PTB-XL
   classes the shared prototypes are enriched for,
4. whether any shared prototype carries signal beyond the known ECG findings of the
   disease (candidates for something new to read off the ECG),
5. compact case-study figures of a prototype's projected windows across datasets.

In [ ]:
import os
os.environ["CACHE_DIR"] = "/home/songs1/.cache/protossl_cache" # for working in dev container

In [ ]:
import collections
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from protossl.datasets import (
    CincECGDataset,
    Code15ECGDataset,
    EchoNextECGDataset,
    HeedbECGDataset,
    MimicECGDataset,
    PtbxlECGDataset,
)
from protossl.defines import (
    CINC_TARGETS,
    CODE15_TARGETS,
    ECHONEXT_TARGETS,
    HEEDB_TARGETS,
    MIMIC_TARGETS,
    PTBXL_TARGETS,
)
from protossl.plotting import (
    chunk_to_seconds,
    get_ecg_waveform,
    plot_segment,
    resolve_index,
)

SEED = 42  # seed for the Venn diagrams, interpretation table and case studies
SEEDS = [42, 67, 70, 73, 99]  # all archived seeds, for the screen and stability

# results root: our archive of the paper's runs, or a new run's outputs, e.g.
# /shared/gpudata/songs1/protossl-shared/runs/<run> (see scripts/README.md)
RUN_DIR = Path(os.environ.get("PROTOSSL_OUTPUTS", "~/protossl-outputs")).expanduser() / "ecg"
PRETRAIN_DIR = RUN_DIR / "protossl-heedb"
MODEL = "protossl-heedb-pila"
PPL = 14

DATA_DIR = Path("/opt/gpudata/ecg")
SAMPLING_RATE = 100
SPLIT = "train"

# the HEEDB row of the case studies needs the HEEDB labels cache, which has to be
# built outside the dev container (its cache dir is read-only); turn on once it exists
INCLUDE_PRETRAIN = False

TEXT_WIDTH = 5.5  # inches, width of the paper's text block

FIG_DIR = Path("figs")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Datasets and labels

`DATASETS` holds the two datasets whose labels we compare. `INTERPRETERS` holds the
classical datasets, used only to read off what a prototype means. `PRETRAIN` is the
HEEDB bank that every dataset draws its prototypes from.

`LABELS` maps a canonical concept onto each dataset's exact target string; the next
cell checks every string against `protossl.defines`. These labels have no HEEDB
pretraining target, which is why there is no HEEDB column. Two mappings are judgement
calls worth knowing about:

- **Aortic valve disease**: phecode 395.2 covers aortic stenosis, regurgitation and
  unspecified aortic valve disorders, so the MIMIC label is broader than EchoNext's
  moderate-or-worse stenosis. EchoNext also has `AR`, which overlaps MIMIC far less.
- **LV systolic dysfunction**: MIMIC also has `Congestive heart failure (chf) nos`
  and `Primary/intrinsic cardiomyopathies`. Both overlap `LVEF Lo` less than HFrEF
  does (see the screen below), as expected for less specific codes.

In [ ]:
PRETRAIN = {
    "slug": "heedb",
    "dataset_cls": HeedbECGDataset,
    "dataset_path": DATA_DIR / "heedb",
    "targets": list(HEEDB_TARGETS),
}

DATASETS = {
    "EchoNext": {
        "slug": "echonext",
        "dataset_cls": EchoNextECGDataset,
        "dataset_path": DATA_DIR / "echonext",
        "targets": list(ECHONEXT_TARGETS),
    },
    "MIMIC": {
        "slug": "mimic",
        "dataset_cls": MimicECGDataset,
        "dataset_path": DATA_DIR / "mimic-iv-ecg",
        "targets": MIMIC_TARGETS,
    },
}

INTERPRETERS = {
    "PTB-XL": {
        "slug": "ptbxl",
        "dataset_cls": PtbxlECGDataset,
        "dataset_path": DATA_DIR / "ptb-xl",
        "targets": PTBXL_TARGETS,
    },
    "CinC": {
        "slug": "cinc",
        "dataset_cls": CincECGDataset,
        "dataset_path": DATA_DIR / "cinc-2020",
        "targets": CINC_TARGETS,
    },
    "CODE-15%": {
        "slug": "code15",
        "dataset_cls": Code15ECGDataset,
        "dataset_path": DATA_DIR / "code15",
        "targets": CODE15_TARGETS,
    },
}

# canonical label -> that label's exact target string in each label set
LABELS = {
    "LV systolic dysfunction": {
        "EchoNext": "LVEF Lo",
        "MIMIC": "Heart failure with reduced ef [systolic or combined heart failure]",
    },
    "Aortic valve disease": {
        "EchoNext": "AS",
        "MIMIC": "Nonrheumatic aortic valve disorders",
    },
}

# filename-safe stems for saved figures
LABEL_SLUGS = {
    "LV systolic dysfunction": "lvsd",
    "Aortic valve disease": "avd",
}

In [ ]:
# every mapped name must exist in the label set it refers to, otherwise a typo
# silently turns into an empty prototype set downstream
for label, mapping in LABELS.items():
    assert label in LABEL_SLUGS, f"no filename slug for {label}"
    assert set(mapping) == set(DATASETS), f"{label}: map every dataset"
    for name, cfg in DATASETS.items():
        assert mapping[name] in cfg["targets"], f"{label}: bad {name} target"

pd.DataFrame(LABELS).T

## Load assignment and projection metadata

- `assignment_metadata.csv` (one row per label slot) says which pretrained prototype
  the ILP gave to each label slot of a dataset. We load it for every seed, since
  the screen and the stability check compare seeds.
- `projection_metadata.csv` (one row per prototype) says which sample and 1-second
  window each prototype was projected onto. We load it only at `SEED`, for plotting.

All seeds share the one HEEDB pretraining run, so prototype indices mean the same
thing in every seed.

In [ ]:
def read_metadata(path: Path) -> pd.DataFrame:
    """Read a metadata CSV, normalizing the sample ID column name.

    `ecg_id` was renamed to `sample_id` when the datasets were generalized beyond
    ECG (see `protossl.trainer.PredictionWriter`), so runs written before that
    rename still carry the old column name. Same with `patient_id` --> `source_id.
    """
    return pd.read_csv(path).rename(columns={"ecg_id": "sample_id", "patient_id": "source_id"})


def run_dir(cfg: dict, seed: int, model: str = MODEL) -> Path:
    return RUN_DIR / f"protossl-outputs-seed{seed}" / f"runs-{cfg['slug']}" / model


heedb_proj = read_metadata(
    PRETRAIN_DIR / "project-prototypes/latest/projection_metadata.csv"
)
N_PROTOTYPES = len(heedb_proj)

ALL_DATASETS = {**DATASETS, **INTERPRETERS}
_datasets = {**ALL_DATASETS, "HEEDB": PRETRAIN}
_dataset_cache: dict[tuple[str, str], object] = {}


def get_dataset(name: str, split: str = SPLIT):
    """Build a dataset once and reuse it, loading waveforms is expensive."""
    if (name, split) not in _dataset_cache:
        cfg = _datasets[name]
        _dataset_cache[(name, split)] = cfg["dataset_cls"](
            dataset_path=str(cfg["dataset_path"]),
            split=split,
            sampling_rate=SAMPLING_RATE,
        )
    return _dataset_cache[(name, split)]


for name, cfg in ALL_DATASETS.items():
    cfg["assign"] = {
        seed: read_metadata(
            run_dir(cfg, seed) / "learn-prototype-assignments/latest/assignment_metadata.csv"
        )
        for seed in SEEDS
    }
    cfg["proj"] = read_metadata(
        run_dir(cfg, SEED) / "project-prototypes-supervised/latest/projection_metadata.csv"
    )
    cfg["ft_proj"] = read_metadata(
        run_dir(cfg, SEED, f"{MODEL}-ft")
        / "project-prototypes-supervised/latest/projection_metadata.csv"
    )

In [ ]:
# the row position of a prototype in `proj`/`ft_proj` is its slot in the projected
# model, and assignment rows are written label-major then slot-minor, so an
# assignment row's position is exactly that slot. the checks below pin down that
# correspondence, which is what lets us index projections by assignment row.
assert (heedb_proj["prototype_id"] == heedb_proj.index).all()

for name, cfg in ALL_DATASETS.items():
    n_slots = len(cfg["targets"]) * PPL
    for seed, assign in cfg["assign"].items():
        assert len(assign) == n_slots, (name, seed)
        assert (assign.index == assign["label_idx"] * PPL + assign["slot_idx"]).all(), (name, seed)
        # ilp_effect_size uses a prototype at most once per dataset
        assert not assign["prototype_idx"].duplicated().any(), (name, seed)
        # prototypes come from the pretrained bank, which HEEDB projection enumerates
        assert assign["prototype_idx"].isin(heedb_proj.index).all(), (name, seed)
    assert len(cfg["proj"]) == len(cfg["ft_proj"]) == n_slots, name
    assert (cfg["proj"]["prototype_id"] == cfg["proj"].index).all(), name
    assert (cfg["ft_proj"]["prototype_id"] == cfg["ft_proj"].index).all(), name
    # the ILP writes labels in the dataset's target order
    labels_in_order = cfg["assign"][SEED].drop_duplicates("label_idx")["label_name"]
    assert list(labels_in_order) == list(cfg["targets"]), name

# by chance, two random sets of PPL prototypes from the bank share PPL^2 / N
CHANCE_OVERLAP = PPL**2 / N_PROTOTYPES
print(f"{N_PROTOTYPES} pretrained prototypes, chance overlap {CHANCE_OVERLAP:.2f} of {PPL}")
pd.Series(
    {name: cfg["assign"][SEED]["prototype_idx"].nunique() for name, cfg in ALL_DATASETS.items()},
    name=f"prototypes used (seed {SEED})",
).to_frame()

## Screen every EchoNext × MIMIC label pair

Overlap between unrelated labels is not zero: some prototypes seem to mark a generally
abnormal ECG and get picked for many labels. So the right null is not
`CHANCE_OVERLAP` but the overlap of all the other label pairs. Each pair's overlap is
averaged over `SEEDS`.

In [ ]:
def label_prototypes(name: str, target: str, seed: int = SEED) -> set[int]:
    """Pretrained prototype indices dataset `name` assigned to `target` at `seed`."""
    assign = ALL_DATASETS[name]["assign"][seed]
    rows = assign.loc[assign["label_name"] == target]
    assert len(rows) == PPL, f"{name} {target}: {len(rows)} slots != {PPL}"
    return set(rows["prototype_idx"])


def overlap_screen(a: str = "EchoNext", b: str = "MIMIC") -> pd.DataFrame:
    """Shared prototype count for every label pair of datasets `a` and `b`, per seed."""
    rows = []
    for seed in SEEDS:
        sets_a = {t: label_prototypes(a, t, seed) for t in ALL_DATASETS[a]["targets"]}
        sets_b = {t: label_prototypes(b, t, seed) for t in ALL_DATASETS[b]["targets"]}
        for (ta, pa), (tb, pb) in itertools.product(sets_a.items(), sets_b.items()):
            rows.append({a: ta, b: tb, "seed": seed, "shared": len(pa & pb)})
    screen = pd.DataFrame(rows).pivot_table(index=[a, b], columns="seed", values="shared")
    screen["mean"] = screen[SEEDS].mean(axis=1)
    screen["rank"] = screen["mean"].rank(ascending=False, method="min").astype(int)
    screen["percentile"] = screen["mean"].rank(pct=True) * 100
    return screen.sort_values("mean", ascending=False)


screen = overlap_screen()
print(
    f"{len(screen)} pairs, mean shared prototypes: median {screen['mean'].median():.1f}, "
    f"90th pct {screen['mean'].quantile(0.9):.1f}, 99th pct {screen['mean'].quantile(0.99):.1f}"
)
screen.head(10)

In [ ]:
# where our label pairs fall in the screen
screen.loc[[(m["EchoNext"], m["MIMIC"]) for m in LABELS.values()]].set_axis(
    list(LABELS), axis=0
)

In [ ]:
# for scale: the same overlap for a classical label, RBBB in PTB-XL and CODE-15%
pd.Series(
    {
        seed: len(label_prototypes("PTB-XL", "CRBBB", seed) & label_prototypes("CODE-15%", "RBBB", seed))
        for seed in SEEDS
    },
    name="RBBB shared, PTB-XL & CODE-15%",
).to_frame().T.assign(mean=lambda df: df.mean(axis=1))

## Stability across seeds

Seeds change the downstream training (subsets, ILP inputs), not the prototype bank.
A prototype that both datasets share at every seed is a property of the bank, not of
one run.

In [ ]:
def shared_prototypes(label: str, seed: int = SEED) -> set[int]:
    """Prototypes every dataset in `DATASETS` assigned to `label` at `seed`."""
    return set.intersection(
        *(label_prototypes(name, LABELS[label][name], seed) for name in DATASETS)
    )


# prototype -> number of seeds at which it is shared, per label
SEEDS_SHARED = {
    label: collections.Counter(p for seed in SEEDS for p in shared_prototypes(label, seed))
    for label in LABELS
}

display(
    pd.DataFrame(
        {label: {seed: len(shared_prototypes(label, seed)) for seed in SEEDS} for label in LABELS}
    ).rename_axis("seed")
)
pd.DataFrame(
    {
        label: pd.Series(counts).value_counts().sort_index(ascending=False)
        for label, counts in SEEDS_SHARED.items()
    }
).fillna(0).astype(int).rename_axis(f"shared at n of {len(SEEDS)} seeds")

## Overlap at `SEED`

The same Venn regions as in `proto-recur-multilabel.ipynb`, now with two datasets.

In [ ]:
def region_table(sets: dict[str, set[int]]) -> pd.DataFrame:
    """Break `sets` into the mutually exclusive regions of a Venn diagram.

    Every region is listed, including empty ones, so that tables for different
    labels line up. Regions are ordered by how many datasets share them.
    """
    names = list(sets)
    rows = []
    for size in range(1, len(names) + 1):
        for combo in itertools.combinations(names, size):
            shared = set.intersection(*(sets[n] for n in combo))
            others = [sets[n] for n in names if n not in combo]
            members = shared.difference(*others) if others else shared
            rows.append(
                {
                    "region": " & ".join(combo) + " only",
                    "n_datasets": size,
                    "count": len(members),
                    "prototypes": sorted(members),
                }
            )
    out = pd.DataFrame(rows)
    assert out["count"].sum() == len(set().union(*sets.values()))
    return out


PROTOTYPES = {
    label: {name: label_prototypes(name, LABELS[label][name]) for name in DATASETS}
    for label in LABELS
}
pd.concat({label: region_table(sets) for label, sets in PROTOTYPES.items()})

In [ ]:
from matplotlib.patches import Patch
from matplotlib_venn import venn2
from matplotlib_venn.layout.venn2 import DefaultLayoutAlgorithm

assert len(DATASETS) == 2, "venn2 draws exactly two sets"

DATASET_COLORS = dict(zip(DATASETS, ["#2a78d6", "#eb6834"]))  # colorblind-safe

with plt.rc_context({"font.size": 7, "axes.titlesize": 8, "axes.titleweight": "bold"}):
    fig, axs = plt.subplots(1, len(PROTOTYPES), figsize=(TEXT_WIDTH * 2 / 3, 1.75 + 0.3))
    for ax, (label, sets) in zip(np.atleast_1d(axs), PROTOTYPES.items()):
        # equal region sizes give every panel the same geometry
        v = venn2(
            [sets[name] for name in DATASETS],
            set_labels=None,
            set_colors=tuple(DATASET_COLORS.values()),
            layout_algorithm=DefaultLayoutAlgorithm(fixed_subset_sizes=(1,) * 3),
            ax=ax,
        )
        ax.set_title(label)
    # take the swatches from the drawn patches so they match the translucent fill
    handles = [
        Patch(facecolor=v.get_patch_by_id(region).get_facecolor(), label=name)
        for region, name in zip(("10", "01"), DATASETS)
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(DATASETS),
        frameon=False,
        bbox_to_anchor=(0.5, 0),
    )
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    fig.savefig(FIG_DIR / f"proto-recur-nonclassical-venn-seed{SEED}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"proto-recur-nonclassical-venn-seed{SEED}.pdf", bbox_inches="tight")

## What the shared prototypes mean in classical datasets

Each classical dataset also assigns each prototype to at most one of its own labels,
so a shared prototype can be read through the label PTB-XL, CinC or CODE-15% gave it.
PTB-XL has 71 labels and uses almost the whole bank, so nearly every prototype has a
PTB-XL reading. That means some readings are leftovers the ILP had to place
somewhere. `association_score` (higher is stronger) shows how well each assignment is
supported, and CinC and CODE-15% assign fewer prototypes, so a reading there is
more selective.

`seeds_shared` counts the seeds at which both datasets share the prototype.

In [ ]:
SCP = pd.read_csv(INTERPRETERS["PTB-XL"]["dataset_path"] / "scp_statements.csv", index_col=0)


def ptbxl_class(code: str) -> str:
    """PTB-XL superclass of a statement, with `form`/`rhythm` for non-diagnostic ones.

    `LPR` (prolonged PR) is a form statement in PTB-XL, so it lands under `form`
    even though it is a conduction finding.
    """
    row = SCP.loc[code]
    if row["diagnostic"] == 1:
        return row["diagnostic_class"]
    return "rhythm" if row["rhythm"] == 1 else "form"


def interpret(label: str, seed: int = SEED) -> pd.DataFrame:
    """Every shared prototype of `label`, with its label in each classical dataset."""
    rows = []
    for proto in sorted(shared_prototypes(label, seed)):
        row = {"prototype": proto, "seeds_shared": SEEDS_SHARED[label][proto]}
        for name in DATASETS:
            assign = ALL_DATASETS[name]["assign"][seed].set_index("prototype_idx")
            row[f"{name} score"] = assign.loc[proto, "association_score"]
        for name, cfg in INTERPRETERS.items():
            assign = cfg["assign"][seed].set_index("prototype_idx")
            if proto in assign.index:
                row[name] = assign.loc[proto, "label_name"]
                row[f"{name} score"] = assign.loc[proto, "association_score"]
        if "PTB-XL" in row:
            row["PTB-XL description"] = SCP.loc[row["PTB-XL"], "description"]
            row["PTB-XL class"] = ptbxl_class(row["PTB-XL"])
        rows.append(row)
    return pd.DataFrame(rows).set_index("prototype")


INTERPRETATIONS = {label: interpret(label) for label in LABELS}
for label, table in INTERPRETATIONS.items():
    print(label)
    display(table.round(2).fillna("-"))

### Enrichment by PTB-XL class

The PTB-XL class of the shared prototypes, compared with the prototypes only one of
the two datasets picked for the label, and with the whole bank. Counts are pooled
over `SEEDS`. The same prototype recurs across seeds, so the pooled counts are not
independent and the table is descriptive, not a test.

In [ ]:
def class_enrichment(label: str) -> pd.DataFrame:
    """Share of each PTB-XL class among shared, one-dataset-only and all prototypes."""
    groups = collections.defaultdict(list)
    for seed in SEEDS:
        ptbxl = INTERPRETERS["PTB-XL"]["assign"][seed].set_index("prototype_idx")["label_name"]
        classes = ptbxl.map(ptbxl_class).reindex(heedb_proj.index).fillna("unassigned")
        sets = [label_prototypes(name, LABELS[label][name], seed) for name in DATASETS]
        shared = set.intersection(*sets)
        groups["shared"] += list(classes[sorted(shared)])
        groups["one dataset only"] += list(classes[sorted(set.union(*sets) - shared)])
        groups["all prototypes"] += list(classes)
    out = pd.DataFrame(
        {group: pd.Series(values).value_counts(normalize=True) for group, values in groups.items()}
    ).fillna(0)
    out.loc["n (pooled)"] = [len(values) for values in groups.values()]
    return out.sort_values("shared", ascending=False)


for label in LABELS:
    print(label)
    display(class_enrichment(label).round(3))

## Beyond known ECG findings

ECG is not how these diseases are diagnosed, and how they show on an ECG is only
partly understood. Everything above shows the shared prototypes line up with known
correlates (old MI, LBBB, LVH, conduction disease). The question here is whether any
of them carry signal those known findings **don't** explain, which is what would make
it a candidate for something new to read off the ECG.

**Design (provisional, to be revised with a cardiologist).** We use the per-ECG
prototype similarities the runs saved (`compute-embeddings`, one column per assigned
slot, at `SEED`).

1. **Incremental value.** A known-findings model (logistic regression on the
   covariates below) is fit on train. On test we ask (a) whether adding the shared
   prototypes raises AUROC, with a bootstrap CI, and (b) per prototype, its odds ratio
   per SD adjusted for the known-findings score (the base model's logit), with
   patient-clustered SEs. `attenuation` is the share of the unadjusted log-odds the
   known findings explain.
2. **ECGs with no known findings.** In the normal-read subset of test, the
   prototype's odds ratio per SD adjusted for age and sex, and its AUROC.

Inference is on **test** because the ILP chose the prototypes using train labels, so
effects estimated on train would be inflated.

**Known findings.**

- Both datasets: age, sex, heart rate, PR, QRS duration, QTc. EchoNext adds the atrial
  rate; MIMIC adds the QRS and T axes, derived from `machine_measurements.csv`.
- MIMIC only: machine-read statements (`KNOWN_STATEMENTS`, matched by regex):
  infarct/Q waves, LBBB, incomplete LBBB, RBBB, IVCD, fascicular block, LVH, left atrial
  abnormality, low voltage, R-wave progression, ST-T abnormality, AF/flutter, paced,
  ectopy, AV block. EchoNext ships no statements, so its adjustment is intervals only,
  a much weaker control. MIMIC is the stringent test.

**Normal-read.**

- MIMIC: no known statement, QRS < 120 ms, PR < 200 ms, heart rate 50–100, QRS axis
  −30 to 90°.
- EchoNext: the same interval criteria, plus atrial rate within 5 bpm of the
  ventricular rate as a stand-in for sinus rhythm with 1:1 conduction.

**Candidate criteria.**

- *Beyond known findings*: adjusted OR > 1 with Holm-corrected p < 0.05 (over all
  prototypes, labels and datasets) in **both** datasets.
- *In normal-read ECGs*: age/sex-adjusted OR > 1 with nominal p < 0.05 in both
  datasets. Only evaluated where the subset has at least `MIN_NORMAL_POS` positives.

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

# machine-read statements counted as known findings, matched case-insensitively
# against all of an ECG's report_* lines
KNOWN_STATEMENTS = {
    "infarct / Q waves": r"infarct|q waves?|\bmi\b",
    "LBBB": r"(?<!incomplete )left bundle branch block|\blbbb\b",
    "incomplete LBBB": r"incomplete left bundle branch block",
    "RBBB": r"right bundle branch block|\brbbb\b",
    "IVCD": r"intraventricular conduction|iv conduction|\bivcd\b",
    "fascicular block": r"fascicular block|\blafb\b|\blpfb\b|hemiblock",
    "LVH": r"left ventricular hypertrophy|\blvh\b",
    "left atrial abnormality": r"left atrial (?:enlargement|abnormality)",
    "low voltage": r"low (?:qrs )?voltage",
    "R-wave progression": r"r[- ]wave progression",
    "ST-T abnormality": r"\bst\b|st-t|t wave|\bt abn|repol|ischemi|injury",
    "AF / flutter": r"atrial fibrillation|atrial flutter",
    "paced": r"pacemaker|paced|pacing",
    "ectopy": r"premature|\bpacs?\b|\bpvcs?\b|ectopic|bigeminy|trigeminy",
    "AV block": r"a-?v block|prolonged pr",
}
MIN_NORMAL_POS = 20  # fewer positives than this in the normal-read subset: not evaluated

MACHINE_MEASUREMENTS = pd.read_csv(
    DATASETS["MIMIC"]["dataset_path"] / "machine_measurements.csv", low_memory=False
).set_index("study_id")


def in_range(x: pd.Series, lo: float, hi: float) -> pd.Series:
    """`x` with values outside [lo, hi] (sentinels like 29999) set to NaN."""
    x = x.astype(float)
    return x.where((x >= lo) & (x <= hi))


def mimic_covariates(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Known findings for MIMIC rows `df`, and which rows are normal-read."""
    mm = MACHINE_MEASUREMENTS.loc[df["study_id"]].reset_index()
    # fiducial points are ms offsets into the recording
    rr = in_range(mm["rr_interval"], 250, 3000)
    cov = pd.DataFrame(
        {
            "age": df["age"].to_numpy(),
            "sex": df["gender"].to_numpy(),
            "heart rate": 60000 / rr,
            "PR": in_range(mm["qrs_onset"] - mm["p_onset"], 60, 400),
            "QRS": in_range(mm["qrs_end"] - mm["qrs_onset"], 40, 250),
            "QTc": in_range(mm["t_end"] - mm["qrs_onset"], 200, 700) / np.sqrt(rr / 1000),
            "QRS axis": in_range(mm["qrs_axis"], -180, 180),
            "T axis": in_range(mm["t_axis"], -180, 180),
        }
    )
    report_cols = [c for c in mm.columns if c.startswith("report_")]
    text = mm[report_cols].fillna("").agg(" | ".join, axis=1).str.lower()
    for finding, pattern in KNOWN_STATEMENTS.items():
        cov[finding] = text.str.contains(pattern, regex=True).astype(int).to_numpy()
    normal = (
        ~cov[list(KNOWN_STATEMENTS)].any(axis=1)
        & (cov["QRS"] < 120)
        & (cov["PR"] < 200)
        & cov["heart rate"].between(50, 100)
        & cov["QRS axis"].between(-30, 90)
    )
    return cov, normal.to_numpy()


def echonext_covariates(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Known findings for EchoNext rows `df` (intervals only), and normal-read rows."""
    cov = pd.DataFrame(
        {
            "age": df["age_at_ecg"].to_numpy(),
            "sex": (df["sex"] == "male").astype(int).to_numpy(),
            "heart rate": in_range(df["ventricular_rate"], 20, 250),
            "atrial rate": in_range(df["atrial_rate"], 20, 600),
            "PR": in_range(df["pr_interval"], 60, 400),
            "QRS": in_range(df["qrs_duration"], 40, 250),
            "QTc": in_range(df["qt_corrected"], 200, 700),
        }
    )
    normal = (
        (cov["QRS"] < 120)
        & (cov["PR"] < 200)
        & cov["heart rate"].between(50, 100)
        & ((cov["atrial rate"] - cov["heart rate"]).abs() <= 5)
    )
    return cov, normal.to_numpy()


COVARIATES = {"EchoNext": echonext_covariates, "MIMIC": mimic_covariates}

In [ ]:
def load_split(name: str, split: str) -> dict:
    """Prototype similarities, labels, known findings and patient IDs of one split."""
    dataset = get_dataset(name, split)
    cfg = ALL_DATASETS[name]
    embeds = np.load(run_dir(cfg, SEED) / f"compute-embeddings/latest/{split}_embeds.npy")
    labels = dataset.labels.numpy()
    assert embeds.shape == (len(labels), len(cfg["targets"]) * PPL), (name, split)
    cov, normal = COVARIATES[name](dataset._df)
    return {
        "embeds": embeds,
        "labels": labels,
        "cov": cov,
        "normal": normal,
        "patients": dataset.source_ids.numpy(),
    }


SPLITS = {(name, split): load_split(name, split) for name in DATASETS for split in ["train", "test"]}

# sanity check: a label's own slots should predict it better than other slots, which
# pins down that embedding rows follow the dataset and columns follow assignment rows
for name in DATASETS:
    test, assign = SPLITS[name, "test"], ALL_DATASETS[name]["assign"][SEED]
    for label, mapping in LABELS.items():
        k = list(ALL_DATASETS[name]["targets"]).index(mapping[name])
        own = assign["label_name"] == mapping[name]
        auc = [roc_auc_score(test["labels"][:, k], test["embeds"][:, j]) for j in range(len(assign))]
        assert np.mean(np.array(auc)[own]) > np.mean(np.array(auc)[~own]) + 0.05, (name, label)

counts = []
for name in DATASETS:
    test = SPLITS[name, "test"]
    for label, mapping in LABELS.items():
        y = test["labels"][:, list(ALL_DATASETS[name]["targets"]).index(mapping[name])]
        counts.append(
            {
                "label": label,
                "dataset": name,
                "test ECGs": len(y),
                "positives": int(y.sum()),
                "normal-read ECGs": int(test["normal"].sum()),
                "normal-read positives": int(y[test["normal"]].sum()),
            }
        )
pd.DataFrame(counts).set_index(["label", "dataset"])

In [ ]:
# prevalence of each known statement among MIMIC test ECGs, to check the regexes
SPLITS["MIMIC", "test"]["cov"][list(KNOWN_STATEMENTS)].mean().rename("prevalence").to_frame().round(3)

In [ ]:
def design_matrix(cov: pd.DataFrame, train_cov: pd.DataFrame) -> pd.DataFrame:
    """Median-impute with train medians, adding a missing indicator where needed."""
    X = cov.copy()
    for col in train_cov.columns:
        if train_cov[col].isna().any():
            X[f"{col} missing"] = X[col].isna().astype(int)
        X[col] = X[col].fillna(train_cov[col].median())
    return X


def logit(y, X, groups):
    """Logistic regression with patient-clustered standard errors."""
    return sm.Logit(y, sm.add_constant(X)).fit(
        disp=0, cov_type="cluster", cov_kwds={"groups": groups}
    )


rng = np.random.default_rng(0)
gain_rows, proto_rows = [], []
for label, mapping in LABELS.items():
    protos = sorted(shared_prototypes(label))
    for name in DATASETS:
        train, test = SPLITS[name, "train"], SPLITS[name, "test"]
        k = list(ALL_DATASETS[name]["targets"]).index(mapping[name])
        y_train, y_test = train["labels"][:, k], test["labels"][:, k]
        X_train = design_matrix(train["cov"], train["cov"])
        X_test = design_matrix(test["cov"], train["cov"])[X_train.columns]
        # each prototype's column is the slot this dataset assigned it to
        cols = [
            int(ALL_DATASETS[name]["assign"][SEED].index[
                ALL_DATASETS[name]["assign"][SEED]["prototype_idx"] == p
            ][0])
            for p in protos
        ]
        mu, sd = train["embeds"][:, cols].mean(0), train["embeds"][:, cols].std(0)
        Z_train = (train["embeds"][:, cols] - mu) / sd
        Z_test = (test["embeds"][:, cols] - mu) / sd

        # step 1a: known findings vs known findings + shared prototypes, out of sample
        known = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
        known.fit(X_train, y_train)
        both = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
        both.fit(np.c_[X_train, Z_train], y_train)
        p_known = known.predict_proba(X_test)[:, 1]
        p_both = both.predict_proba(np.c_[X_test, Z_test])[:, 1]
        deltas = []
        for _ in range(1000):
            i = rng.integers(0, len(y_test), len(y_test))
            if y_test[i].min() != y_test[i].max():
                deltas.append(roc_auc_score(y_test[i], p_both[i]) - roc_auc_score(y_test[i], p_known[i]))
        gain_rows.append(
            {
                "label": label,
                "dataset": name,
                "AUROC known": roc_auc_score(y_test, p_known),
                "AUROC known + shared": roc_auc_score(y_test, p_both),
                "ΔAUROC": roc_auc_score(y_test, p_both) - roc_auc_score(y_test, p_known),
                "95% CI lo": np.percentile(deltas, 2.5),
                "95% CI hi": np.percentile(deltas, 97.5),
            }
        )

        # step 1b and step 2, one prototype at a time, on test
        known_score = known.decision_function(X_test)
        normal = test["normal"]
        n_normal_pos = int(y_test[normal].sum())
        age_sex = test["cov"].loc[normal, ["age", "sex"]].to_numpy(float)
        for j, proto in enumerate(protos):
            z = Z_test[:, j]
            unadj = logit(y_test, z, test["patients"])
            adj = logit(y_test, np.c_[known_score, z], test["patients"])
            row = {
                "label": label,
                "dataset": name,
                "prototype": proto,
                "OR unadjusted": np.exp(unadj.params[1]),
                "OR adjusted": np.exp(adj.params[2]),
                "OR adj lo": np.exp(adj.conf_int()[2, 0]),
                "OR adj hi": np.exp(adj.conf_int()[2, 1]),
                "p adjusted": adj.pvalues[2],
                "attenuation": 1 - adj.params[2] / unadj.params[1],
                "normal-read positives": n_normal_pos,
            }
            if n_normal_pos >= MIN_NORMAL_POS:
                nr = logit(y_test[normal], np.c_[age_sex, z[normal]], test["patients"][normal])
                row |= {
                    "normal-read AUROC": roc_auc_score(y_test[normal], z[normal]),
                    "normal-read OR": np.exp(nr.params[3]),
                    "normal-read p": nr.pvalues[3],
                }
            proto_rows.append(row)

GAINS = pd.DataFrame(gain_rows).set_index(["label", "dataset"])
BEYOND = pd.DataFrame(proto_rows)
BEYOND["p adjusted (Holm)"] = multipletests(BEYOND["p adjusted"], method="holm")[1]
BEYOND = BEYOND.set_index(["label", "prototype", "dataset"]).sort_index()
GAINS.round(3)

In [ ]:
BEYOND.round(3)

In [ ]:
def candidates() -> pd.DataFrame:
    """Which shared prototypes meet each provisional criterion in both datasets.

    `None` in the normal-read column means some dataset had too few positives in
    its normal-read subset to evaluate.
    """
    wide = BEYOND.unstack("dataset")
    beyond = ((wide["OR adjusted"] > 1) & (wide["p adjusted (Holm)"] < 0.05)).all(axis=1)
    if "normal-read OR" in wide:
        evaluable = wide["normal-read OR"].notna().all(axis=1)
        passes = ((wide["normal-read OR"] > 1) & (wide["normal-read p"] < 0.05)).all(axis=1)
        normal = passes.astype(object).where(evaluable, None)
    else:
        normal = pd.Series(None, index=wide.index, dtype=object)
    ptbxl = pd.concat(
        {label: INTERPRETATIONS[label]["PTB-XL"] for label in LABELS}, names=["label", "prototype"]
    )
    return pd.DataFrame(
        {
            "PTB-XL reading": ptbxl.reindex(wide.index),
            "beyond known findings": beyond,
            "in normal-read ECGs": normal,
        }
    )


candidates()

### Reading these results

- **Effects can point either way.** The ILP assigns by effect size, so a prototype
  can be negatively associated with its label (OR < 1). Such prototypes don't count
  as candidates under the criteria above, but they're still informative.
- **EchoNext vs MIMIC.** Without statements, EchoNext's known-findings model misses
  infarct, LBBB, LVH and so on. A prototype can therefore look "beyond known" in
  EchoNext just by encoding one of those, which is why the criterion requires MIMIC
  too.
- **MIMIC's normal-read subset barely contains these diseases.** A MIMIC ED patient
  with HFrEF or aortic valve disease almost always has some machine-read finding, so
  the subset is too small to test (see the counts above). Testing it properly needs
  statement-level known findings in EchoNext, e.g. PTB-XL's labels imputed through
  its linear probe. That requires similarities to the full prototype bank on EchoNext
  ECGs, a GPU `compute-embeddings` run.

## Case studies: projected windows only

As in `proto-recur-multilabel.ipynb`, each run projects a prototype onto the training
sample and 1-second window that activate it most, so each run has its own source
waveform. Rows are runs, grouped by dataset; columns are the leads chosen per case.
The classical dataset rows show the projection for the label that dataset assigned
the prototype to (in the group name), which is the ECG reading of the prototype.

`has_label` checks that each source ECG carries the label that run projected for. The
HEEDB row (`INCLUDE_PRETRAIN`) is a label-free projection, so it has no label to check.

Pick prototypes from the tables above: known correlates are shared at every seed and
have strong scores in a classical dataset, and candidates come from `candidates()`.

In [ ]:
RUN_NAMES = {"pretrain": "pretrained", "probe": "linear probe", "ft": "fine-tuned"}
CASE_INTERPRETERS = ["PTB-XL"]  # classical datasets to show, each adds two rows


def prototype_sources(
    proto_idx: int,
    interpreters: list[str] = CASE_INTERPRETERS,
    include_pretrain: bool = INCLUDE_PRETRAIN,
) -> pd.DataFrame:
    """Sample and window each run projected prototype `proto_idx` onto, at `SEED`.

    One row per run: the HEEDB pretraining projection (optional), then the linear
    probe and fine-tuned projections of every dataset (and interpreter) that
    assigned the prototype, with the label it was assigned to.
    """
    rows = []
    if include_pretrain:
        rows.append(
            {"dataset": "HEEDB", "run": "pretrain", "label": None, **heedb_proj.loc[proto_idx]}
        )
    for name in [*DATASETS, *interpreters]:
        cfg = ALL_DATASETS[name]
        assign = cfg["assign"][SEED]
        slots = assign.index[assign["prototype_idx"] == proto_idx]
        if len(slots) == 0:
            continue  # this dataset did not use the prototype for any label
        slot = slots[0]  # at most one, checked when loading
        target = assign.loc[slot, "label_name"]
        rows.append({"dataset": name, "run": "probe", "label": target, **cfg["proj"].loc[slot]})
        rows.append({"dataset": name, "run": "ft", "label": target, **cfg["ft_proj"].loc[slot]})
    columns = ["dataset", "run", "label", "source_id", "sample_id", "chunk_idx", "emb_sim"]
    ids = ["source_id", "sample_id", "chunk_idx"]  # floated by the mixed-dtype rows
    return pd.DataFrame(rows)[columns].astype(dict.fromkeys(ids, int))


def prototype_windows(
    proto_idx: int, split: str = SPLIT, **kwargs
) -> tuple[pd.DataFrame, list[np.ndarray]]:
    """Projected 1-second (L, T) window of `proto_idx` in every run that uses it.

    Also flags whether each window's source ECG carries the label of its run.
    """
    sources = prototype_sources(proto_idx, **kwargs)
    windows, has_label = [], []
    for _, row in sources.iterrows():
        dataset = get_dataset(row["dataset"], split=split)
        idx = resolve_index(dataset, sample_id=int(row["sample_id"]))
        signal = get_ecg_waveform(
            dataset=dataset, index=idx, sampling_rate=SAMPLING_RATE
        )
        # same windowing as plot_ecg's chunk_idx highlight
        start, end = chunk_to_seconds(
            int(row["chunk_idx"]),
            chunk_len=SAMPLING_RATE,
            chunk_overlap=0.5,
            sampling_rate=SAMPLING_RATE,
        )
        windows.append(signal[:, int(start * SAMPLING_RATE) : int(end * SAMPLING_RATE)])
        if row["label"] is None:
            has_label.append(None)  # label-free pretraining projection
        else:
            targets = list(_datasets[row["dataset"]]["targets"])
            has_label.append(bool(dataset[idx]["label"][targets.index(row["label"])]))
    return sources.assign(has_label=has_label), windows

In [ ]:
def plot_prototype_windows(
    sources: pd.DataFrame,
    windows: list[np.ndarray],
    leads: list[str],
    row_height: float = 0.8,
) -> plt.Figure:
    """Grid of projected windows, one row per run and one column per lead.

    Classical datasets are named with the label they assigned the prototype to.
    """
    group_names = {
        dataset: f"{dataset}: {label}" if dataset in INTERPRETERS else dataset
        for dataset, label in zip(sources["dataset"], sources["label"])
    }
    # one grid row per run, plus a thin spacer row wherever the dataset changes
    height_ratios, grid_rows = [], []
    for i, dataset in enumerate(sources["dataset"]):
        if i > 0 and dataset != sources["dataset"].iloc[i - 1]:
            height_ratios.append(0.25)
        grid_rows.append(len(height_ratios))
        height_ratios.append(1)

    with plt.rc_context({"font.size": 7}):
        fig = plt.figure(figsize=(TEXT_WIDTH, row_height * sum(height_ratios) + 0.2))
        gs = fig.add_gridspec(
            len(height_ratios),
            len(leads),
            height_ratios=height_ratios,
            hspace=0.12,
            wspace=0.06,
        )
        group_axs: dict[str, list[plt.Axes]] = {}
        for (_, row), grid_row, window in zip(sources.iterrows(), grid_rows, windows):
            axs = np.array([[fig.add_subplot(gs[grid_row, j]) for j in range(len(leads))]])
            plot_segment(
                window,
                axs,
                sampling_rate=SAMPLING_RATE,
                lead_layout=[leads],
                label_leads=False,  # labeled once as column headers instead
            )
            axs[0, 0].set_ylabel(
                RUN_NAMES[row["run"]], rotation=0, ha="right", va="center", labelpad=4
            )
            group_axs.setdefault(row["dataset"], []).append(axs[0, 0])
            if grid_row == 0:
                for ax, lead in zip(axs.flat, leads):
                    ax.set_title(lead, fontweight="bold", fontsize=8)

        # dataset names go left of the run names, centered on each group's rows
        fig.canvas.draw()
        to_fig = fig.transFigure.inverted()
        run_label_x0 = min(
            ax.yaxis.label.get_window_extent().x0
            for axs in group_axs.values()
            for ax in axs
        )
        x = to_fig.transform((run_label_x0, 0))[0] - 0.015
        for dataset, axs in group_axs.items():
            y = (axs[0].get_position().y1 + axs[-1].get_position().y0) / 2
            fig.text(
                x, y, group_names[dataset], rotation=90, ha="right", va="center",
                fontweight="bold", fontsize=8,
            )
    return fig

In [ ]:
# (concept, prototype, leads): leads chosen for the prototype's classical reading
CASE_STUDIES = [
    # PTB-XL: ST-T changes of ventricular aneurysm, i.e. old anterior MI
    ("LV systolic dysfunction", 217, ["V1", "V2", "V3", "V4"]),
    # PTB-XL: incomplete LBBB, CODE-15%: LBBB
    ("LV systolic dysfunction", 991, ["I", "V1", "V5", "V6"]),
    # PTB-XL: LVH
    ("Aortic valve disease", 728, ["aVL", "V1", "V5", "V6"]),
    # PTB-XL: prolonged PR, CinC: 1st-degree AV block, CODE-15%: 1st-degree AV block
    ("Aortic valve disease", 992, ["I", "II", "V1", "V5"]),
]

for label, proto_idx, leads in CASE_STUDIES:
    assert proto_idx in shared_prototypes(label), f"{proto_idx} not shared for {label}"
    sources, windows = prototype_windows(proto_idx)
    print(f"{label} | prototype {proto_idx}")
    display(sources)
    assert sources["has_label"].dropna().all()

    fig = plot_prototype_windows(sources, windows, leads)
    stem = f"{LABEL_SLUGS[label]}-p{proto_idx}-windows-seed{SEED}"
    fig.savefig(FIG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()